# Jev on the SEC firehose

Live demo · AI Tinkerers Toronto · Nov 26, 2026

**The problem:** we want the same piece of data from thousands of contracts.
**The data here:** does the contract change if the company is sold? Yes or no.

1. **Ask** one question about one contract.
2. **Check** Jev and a GPT model against lawyers' labels.
3. **Run** every contract through both models, and watch the cost.

## Setup

Line 3 picks the GPT model: `gpt-6-luna` for rehearsal, `gpt-6.2-astra` onstage.

In [1]:
# --- ONSTAGE MODEL SWITCH ---
# Rehearsal default: "gpt-6-luna"  |  Onstage switch: "gpt-6.2-astra"
LLM_MODEL = "gpt-6-luna"
# LLM_MODEL = "gpt-6.2-astra"

import asyncio
import json
import os
import textwrap
import time
from pathlib import Path

import openai
from IPython.display import clear_output
from pydantic import BaseModel
from typesafe_sdk import AsyncTypeSafeClient, Noul, RetryPolicy

missing = [key for key in ("OPENAI_API_KEY", "TYPESAFE_API_KEY") if not os.environ.get(key)]
if missing:
    raise RuntimeError(
        f"Missing {', '.join(missing)}. Launch with:\n"
        "  with-secrets OPENAI_API_KEY TYPESAFE_API_KEY -- "
        'env VIRTUAL_ENV="$HOME/pro/jev-intro-to-system-one-ai/.venv" nvb demo.ipynb'
    )

jev = AsyncTypeSafeClient(retry=RetryPolicy(max_retries=4))
gpt = openai.AsyncOpenAI()
print(f"Ready: Jev and {LLM_MODEL}")

Ready: Jev and gpt-6-luna


## 1 · One contract, one question

Load the contracts downloaded from this month's SEC filings and look at one of them.

In [2]:
days = sorted(Path("data/exhibits").glob("*.jsonl"))
month = [json.loads(line) for day in days for line in day.open()]
if not month:
    raise FileNotFoundError("No contracts in data/exhibits/. Download them first: python edgar.py 2026-11")

NEXSTAR = "https://www.sec.gov/Archives/edgar/data/1142417/000119312526477077/nxst-ex10_1.htm"
contract = {c["url"]: c for c in month}[NEXSTAR]
at = contract["text"].find("Change in Control")
excerpt = " ".join(contract["text"][at - 120 : at + 340].split(" ")[1:-1])

print(f"{len(month):,} contracts from {len(days)} filing days. Here is one of them:\n")
print(f"{contract['title']}  ({len(contract['text']):,} characters)\n")
print(textwrap.fill(f"...{excerpt}...", width=88))

750 contracts from 17 filing days. Here is one of them:

NEXSTAR MEDIA GROUP, INC.  (48,512 characters)

...Executive’s employment for any reason or for no reason, other than for Cause and
including in connection with a Change in Control (as defined in Paragraph 21(d)), upon
thirty (30) days prior written notice to Executive, subject to payment of the
termination payments specified in Paragraph 6 hereof. Such termination will be effective
as of the date stated in a written notice of termination delivered by the Company to
Executive. (e) Termination by...


Ask Jev one question about it. The answer is one number: the probability of yes.

In [3]:
QUESTION = (
    "Does this contract give a party the right to terminate, withhold consent, or modify "
    "terms if a party undergoes a change of control (merger, acquisition, ownership change, "
    "or sale of substantially all assets)?"
)


async def ask_jev(text):
    reply = await jev.system_one(state=text, questions={"answer": Noul(instructions=QUESTION)})
    return reply.answers["answer"].noul, reply.usage


probability, usage = await ask_jev(contract["text"])
print(f"Jev: {probability:.0%} probability of yes")

Jev: 60% probability of yes


What did that one answer cost? Tokens times the price.

In [4]:
# Dollars per million tokens: (input, output)
PRICES = {"jev": (0.042, 0.00), "gpt-6-luna": (0.10, 0.50), "gpt-6.2-astra": (8.00, 40.00)}


def cost(model, usage):
    price_in, price_out = PRICES[model]
    return (usage.input_tokens * price_in + usage.output_tokens * price_out) / 1_000_000


print(f"{usage.input_tokens:,} tokens in, {usage.output_tokens:,} out: ${cost('jev', usage):.5f}")

10,219 tokens in, 20 out: $0.00043


Now the same contract and the same question for the GPT model. It answers yes or no.

In [6]:
class Verdict(BaseModel):
    answer: bool


async def ask_llm(text):
    reply = await gpt.responses.parse(
        model=LLM_MODEL,
        input=f"Answer strictly from the contract text.\n\nQuestion: {QUESTION}\n\nContract text:\n{text}",
        text_format=Verdict,
        reasoning={"effort": "low"},
    )
    return reply.output_parsed.answer, reply.usage


verdict, llm_usage = await ask_llm(contract["text"])
print(f"{LLM_MODEL}: {'yes' if verdict else 'no'}")
print(f"{llm_usage.input_tokens:,} tokens in, {llm_usage.output_tokens:,} out: ${cost(LLM_MODEL, llm_usage):.5f}")

gpt-6-luna: yes
9,575 tokens in, 112 out: $0.00101


Jev can answer several questions in one call, and an answer can be a score.
Add a second question that rates the contract on a scale.

In [7]:
from typesafe_sdk import Score

reply = await jev.system_one(
    state=contract["text"],
    questions={
        "change_of_control": Noul(instructions=QUESTION),
        "buyer_risk": Score(
            instructions="How much risk does this contract carry for someone who buys the company?",
            criteria=["No risk", "Low risk", "Medium risk", "High risk"],
        ),
    },
)
print(f"Change of control: {reply.answers['change_of_control'].noul:.0%} probability of yes")
print(f"Buyer risk: {reply.answers['buyer_risk'].score:.1f} on a scale from 0 (no risk) to 3 (high risk)")
print(f"Both answers: ${cost('jev', reply.usage):.5f}")

Change of control: 56% probability of yes
Buyer risk: 1.8 on a scale from 0 (no risk) to 3 (high risk)
Both answers: $0.00043


## 2 · Check against lawyers' labels

Eight contracts where lawyers already marked the answer (the CUAD dataset).
Jev and the GPT model get the same contracts and the same question. The models never see the labels.

In [8]:
labeled = json.loads(Path("data/cuad-sample.json").read_text())
for model, ask in (("jev", ask_jev), (LLM_MODEL, ask_llm)):
    start = time.perf_counter()
    replies = await asyncio.gather(*(ask(c["text"]) for c in labeled))
    seconds = time.perf_counter() - start
    # Jev returns a probability and the GPT model returns True or False. 0.5 or more counts as yes.
    right = sum((answer >= 0.5) == c["gold"]["change_of_control"] for (answer, _), c in zip(replies, labeled))
    spent = sum(cost(model, usage) for _, usage in replies)
    print(f"{model:<14} {right} of {len(labeled)} match the lawyers   {seconds:5.1f} s   ${spent:.4f}")

jev            8 of 8 match the lawyers     0.2 s   $0.0025
gpt-6-luna     7 of 8 match the lawyers     2.7 s   $0.0057


## 3 · Every contract, live

Every contract goes to both models at the same time.
Each answer is written to a file the moment it arrives, so stopping early keeps everything pulled so far.

Stop it with `ii`. A model also stops on its own when it reaches the spend limit.

In [9]:
SPEND_LIMIT = 10.00  # dollars per model. A model that reaches it stops.
WORKERS = 8  # requests in flight per model

run_dir = Path("runs") / time.strftime("%Y%m%d-%H%M%S")
run_dir.mkdir(parents=True)
tally = {model: {"answered": 0, "failed": 0, "spent": 0.0, "seconds": 0.0} for model in ("jev", LLM_MODEL)}
started = time.perf_counter()

In [10]:
async def run_model(model, ask):
    """Ask one model about every contract, saving each answer as it arrives."""
    todo, count = iter(month), tally[model]
    with open(run_dir / f"{model}.jsonl", "w") as saved:

        async def worker():
            for contract in todo:
                if count["spent"] >= SPEND_LIMIT:
                    return
                row = {"id": contract["id"]}
                try:
                    answer, usage = await ask(contract["text"])
                    row |= {"answer": answer, "usd": cost(model, usage)}
                    count["spent"] += row["usd"]
                    count["answered"] += 1
                except Exception as error:
                    row["error"] = count["error"] = repr(error)
                    count["failed"] += 1
                saved.write(json.dumps(row) + "\n")
                saved.flush()
                count["seconds"] = time.perf_counter() - started

        await asyncio.gather(*(worker() for _ in range(WORKERS)))


def clock(seconds):
    return f"{seconds / 3600:.1f} hours" if seconds >= 3600 else f"{seconds / 60:.1f} min"


def show(note):
    lines = [
        f"{len(month):,} contracts   {time.perf_counter() - started:.0f} s elapsed   {note}",
        "",
        f"{'model':<15}{'answered':>10}{'spent':>12}    all {len(month):,} at this pace",
    ]
    for model, count in tally.items():
        pace = "waiting for the first answer"
        if count["answered"]:
            scale = len(month) / count["answered"]
            pace = f"${count['spent'] * scale:,.2f} and {clock(count['seconds'] * scale)}"
        lines.append(f"{model:<15}{count['answered']:>10,}{'$' + format(count['spent'], ',.4f'):>12}    {pace}")
        if count["failed"]:
            lines.append(f"{'':<15}{count['failed']:,} failed. Latest: {count['error']}")
    clear_output(wait=True)
    print("\n".join(lines))


jobs = [asyncio.create_task(run_model("jev", ask_jev)), asyncio.create_task(run_model(LLM_MODEL, ask_llm))]
try:
    while not all(job.done() for job in jobs):
        show(f"spend limit ${SPEND_LIMIT:.2f} per model   ii to stop")
        await asyncio.sleep(0.5)
    await asyncio.gather(*jobs)
    show("finished")
except asyncio.CancelledError:
    for job in jobs:
        job.cancel()
    await asyncio.gather(*jobs, return_exceptions=True)
    show("stopped by hand")
print(f"\nEvery answer so far is saved in {run_dir}/")

750 contracts   97 s elapsed   finished

model            answered       spent    all 750 at this pace
jev                   750     $0.1742    $0.17 and 0.2 min
gpt-6-luna            750     $0.3878    $0.39 and 1.6 min

Every answer so far is saved in runs/20261126-191227/


The saved files are the result. Read them back: this is the review queue.

In [6]:
saved = {path.stem: [json.loads(line) for line in path.open()] for path in sorted(run_dir.glob("*.jsonl"))}
answers = {model: {row["id"]: row["answer"] for row in rows if "answer" in row} for model, rows in saved.items()}
for model, found in answers.items():
    print(f"{model:<14} {len(found):,} answers kept in {run_dir / (model + '.jsonl')}")

both = answers["jev"].keys() & answers[LLM_MODEL].keys()
agree = sum((answers["jev"][i] >= 0.5) == answers[LLM_MODEL][i] for i in both)
print(f"\nThe two models agree on {agree} of {len(both)} contracts that both answered.")

print("\nReview queue: Jev's ten highest scores")
titles = {c["id"]: c["title"] for c in month}
for contract_id in sorted(answers["jev"], key=answers["jev"].get, reverse=True)[:10]:
    print(f"  {answers['jev'][contract_id]:>4.0%}  {titles[contract_id]}")

KeyError: 'jev'